In [1]:
import torch
import onnx
from finn.util.test import get_test_model_trained
from brevitas.export import export_qonnx
from qonnx.util.cleanup import cleanup as qonnx_cleanup
from qonnx.core.modelwrapper import ModelWrapper
from finn.transformation.qonnx.convert_qonnx_to_finn import ConvertQONNXtoFINN
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs
import os
# os.chdir('/mnt/c/Users/An/finn/notebooks/sin_pulse')

build_dir = "."

export_onnx_path = build_dir + "/finn_ready_model.onnx"
qonnx_cleanup(export_onnx_path, out_file=export_onnx_path)
model = ModelWrapper(export_onnx_path)
model = model.transform(ConvertQONNXtoFINN())
model = model.transform(InferShapes())
model = model.transform(FoldConstants())
model = model.transform(GiveUniqueNodeNames())
model = model.transform(GiveReadableTensorNames())
model = model.transform(RemoveStaticGraphInputs())
model.save(build_dir + "/2dconv_model_pre.onnx")

In [2]:
from finn.transformation.streamline import Streamline
from qonnx.transformation.lower_convs_to_matmul import LowerConvsToMatMul
from qonnx.transformation.bipolar_to_xnor import ConvertBipolarMatMulToXnorPopcount
import finn.transformation.streamline.absorb as absorb
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from qonnx.transformation.general import RemoveUnusedTensors

model = ModelWrapper(build_dir + "/2dconv_model_pre.onnx")
model = model.transform(MoveScalarLinearPastInvariants())
model = model.transform(Streamline())
model = model.transform(LowerConvsToMatMul())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(absorb.AbsorbTransposeIntoMultiThreshold())
# model = model.transform(ConvertBipolarMatMulToXnorPopcount())
model = model.transform(Streamline())
# absorb final add-mul nodes into TopK
# model = model.transform(absorb.AbsorbScalarMulAddIntoTopK())
model = model.transform(InferDataLayouts())
model = model.transform(RemoveUnusedTensors())
model.save(build_dir + "/2dconv_model_streamlined.onnx")

/home/cutesquare/finn/deps/qonnx/src/qonnx/transformation/infer_data_layouts.py:127: UserWarning: Assuming 4D input is NCHW
  warnings.warn("Assuming 4D input is NCHW")


In [3]:
from finn.util.basic import pynq_part_map
# change this if you have a different PYNQ board, see list above

import finn.transformation.fpgadataflow.convert_to_hw_layers as to_hw
from finn.transformation.fpgadataflow.create_dataflow_partition import (
    CreateDataflowPartition,
)
from finn.transformation.move_reshape import RemoveCNVtoFCFlatten
from finn.transformation.fpgadataflow.specialize_layers import SpecializeLayers
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.infer_data_layouts import InferDataLayouts
from finn.transformation.streamline.reorder import MakeMaxPoolNHWC, MoveScalarLinearPastInvariants


model = ModelWrapper(build_dir + "/2dconv_model_streamlined.onnx")
model = model.transform(to_hw.InferBinaryMatrixVectorActivation())
model = model.transform(to_hw.InferQuantizedMatrixVectorActivation())
# TopK to LabelSelect
# model = model.transform(to_hw.InferLabelSelectLayer())
# input quantization (if any) to standalone thresholding
model = model.transform(to_hw.InferThresholdingLayer())
model = model.transform(to_hw.InferConvInpGen())
model = model.transform(MakeMaxPoolNHWC())
model = model.transform(to_hw.InferStreamingMaxPool())
# get rid of Reshape(-1, 1) operation between hw nodes
model = model.transform(RemoveCNVtoFCFlatten())
# get rid of Tranpose -> Tranpose identity seq
model = model.transform(absorb.AbsorbConsecutiveTransposes())
# infer tensor data layouts
model = model.transform(InferDataLayouts())
parent_model = model.transform(CreateDataflowPartition())
parent_model.save(build_dir + "/2dconv_model_dataflow_parent.onnx")
sdp_node = parent_model.get_nodes_by_op_type("StreamingDataflowPartition")[0]
sdp_node = getCustomOp(sdp_node)
dataflow_model_filename = sdp_node.get_nodeattr("model")
# save the dataflow partition with a different name for easier access
# and specialize the layers to HLS variants
dataflow_model = ModelWrapper(dataflow_model_filename)
# dataflow_model = dataflow_model.transform(SpecializeLayers(fpga_part))
dataflow_model.save(build_dir + "/2dconv_model_dataflow_model.onnx")

In [4]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
import os
import shutil
# os.chdir('/mnt/c/Users/An/finn/notebooks/sin_pulse')

## Build flow with hardware build
build_dir = "."
# model_file = build_dir+"/2dconv_model_folded.onnx"
model_file = build_dir+"/2dconv_model_dataflow_model.onnx"

output_dir = build_dir + "/output"

#Delete previous run results if exist
if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print("Previous run results deleted!")

build_steps = [
    "step_qonnx_to_finn",
    "step_tidy_up",
    "step_streamline",
    "step_convert_to_hw",
    "step_create_dataflow_partition",
    "step_specialize_layers",
    "step_apply_folding_config",
    "step_minimize_bit_width",
    "step_generate_estimate_reports",
    "step_specialize_layers",
    # "step_target_fps_parallelization",
    "step_minimize_bit_width",
    "step_generate_estimate_reports",
    "step_hw_codegen",
    "step_hw_ipgen",
    "step_set_fifo_depths",
    "step_create_stitched_ip",
    "step_measure_rtlsim_performance",
    # "step_out_of_context_synthesis",
    # "step_synthesize_bitfile",
    # "step_make_pynq_driver",
]

cfg_build = build.DataflowBuildConfig(
    output_dir                    = output_dir,
    mvau_wwidth_max     = 80,
    synth_clk_period_ns = 20.0,
    auto_fifo_depths = False,
    #specialize_layers_config_file = "specialize_layers_all_hls.json",
    folding_config_file           = "final_hw_config.json",
    fpga_part                         = "xc7z010clg400-1",
    # shell_flow_type               = build_cfg.ShellFlowType.VIVADO_ZYNQ,
    steps                         = build_steps,
    default_swg_exception         = True,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
        build_cfg.DataflowOutputType.RTLSIM_PERFORMANCE,
    ],
)



Previous run results deleted!


In [5]:
import os
# os.chdir('/mnt/c/Users/An/finn/notebooks/sin_pulse')
build.build_dataflow_cfg(model_file, cfg_build)

Building dataflow accelerator from ./2dconv_model_dataflow_model.onnx
Intermediate outputs will be generated in /tmp/finn_dev_cutesquare
Final outputs will be generated in ./output
Build log is at ./output/build_dataflow.log
Running step: step_qonnx_to_finn [1/17]
Running step: step_tidy_up [2/17]
Running step: step_streamline [3/17]
Running step: step_convert_to_hw [4/17]
Running step: step_create_dataflow_partition [5/17]
Running step: step_specialize_layers [6/17]
Running step: step_apply_folding_config [7/17]
Running step: step_minimize_bit_width [8/17]
Running step: step_generate_estimate_reports [9/17]
Running step: step_specialize_layers [10/17]
Running step: step_minimize_bit_width [11/17]
Running step: step_generate_estimate_reports [12/17]
Running step: step_hw_codegen [13/17]
Running step: step_hw_ipgen [14/17]
Running step: step_set_fifo_depths [15/17]
Running step: step_create_stitched_ip [16/17]
Running step: step_measure_rtlsim_performance [17/17]
Completed successfully


0

In [6]:
os.environ['FINN_ROOT']

'/home/cutesquare/finn'